# Relay: vLLM benchmark on the Colab T4

Runs the three Colab configurations of [ADR-001](https://github.com/sanskar800/relay-llm-gateway/blob/main/docs/adr/001-inference-engines-and-hardware.md)
through Relay, the same open-loop harness and workload as the local runs:

| Config | Model | Quantisation |
|---|---|---|
| `colab-t4-1.5b-fp16` | Qwen2.5-1.5B-Instruct | FP16 |
| `colab-t4-1.5b-awq` | Qwen2.5-1.5B-Instruct-AWQ | AWQ 4-bit |
| `colab-t4-7b-awq` | Qwen2.5-7B-Instruct-AWQ | AWQ 4-bit |

**How to use:** `Runtime > Change runtime type > T4 GPU` (preselected), then `Runtime > Run all`.
About 60-75 minutes. **Do not edit code here**: everything comes from the repository
(`scripts/colab_run.sh`, `bench/scenarios/colab-t4-*.yaml`), so a run can be reproduced from its commit.

Each configuration: start vLLM -> start Relay (standalone: no Postgres/Redis) -> probe to find
saturation -> open-loop run at 0.25x-1.25x of the probe's peak -> zip `results/`. Results are
zipped **after every configuration** (and copied to Google Drive if mounted), because free Colab
sessions can end without warning. At the end the full zip downloads; commit it under `results/`.

## 1. Check the GPU
The scenarios are labelled *Tesla T4*; any other GPU would mislabel the results, so the notebook stops.

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu or "no GPU")
assert "T4" in gpu, (
    "Not a T4: set Runtime > Change runtime type > T4 GPU (results are labelled T4)."
)

## 2. Google Drive (optional, recommended)
Copies each configuration's zip to `MyDrive/relay-results/` as soon as it finishes, so a disconnect loses at most the configuration in progress. Approve the pop-up, or skip this cell.

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as e:  # not fatal: zips are also kept in /content
    print("Drive not mounted:", e)

## 3. Get the code and install
`uv` installs Relay and the harness (Python 3.12, exact versions from `uv.lock`); vLLM is pinned to the version checked on Day 2. About 5 minutes.

In [ ]:
import os

REPO = "https://github.com/sanskar800/relay-llm-gateway.git"
REF = "main"  # a commit SHA here pins the exact code; env.json records the SHA either way
%cd /content
!rm -rf relay && git clone -q $REPO relay
%cd /content/relay
!git checkout -q $REF && git log -1 --format="code at %H (%cd)"
assert os.path.exists("scripts/colab_run.sh"), (
    f"scripts/colab_run.sh is missing at {REF}: the benchmark code is not on GitHub yet."
)

In [ ]:
!pip install -q uv
!uv sync -q --frozen
!pip install -q vllm==0.31.0
!.venv/bin/python -c "import relay, bench; print('relay + harness OK')"
!vllm --version

## 4. Run the configurations
One cell each, so a failure in one does not stop the others. Each prints its probe, the rates it chose, the per-rate summary, and where the zip went. A line starting `INVALID RUN` means that configuration's results must not be used.

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-1.5b-fp16

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-1.5b-awq

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-7b-awq

## 4b. Answer quality: GSM8K (about 3-5 min per configuration)
The same 200 maths questions for every configuration, temperature 0, scored by exact final number (`bench/quality/gsm8k.py`). Answers whether a faster configuration gives worse answers. Independent of section 4: run these alone if the speed results exist.

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-1.5b-fp16 --quality

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-1.5b-awq --quality

In [ ]:
%cd /content/relay
!bash scripts/colab_run.sh colab-t4-7b-awq --quality

## 5. Summary and download
A quick look only (the report is generated later from the Parquet files), then the zip of everything downloads. Send it back to be committed under `results/`.

In [ ]:
import json
import pathlib

for q in sorted(pathlib.Path("results/quality").glob("gsm8k-*")):
    s = json.loads((q / "summary.json").read_text())["summary"]
    low, high = s["ci95"]
    print(f"{q.name}: {s['accuracy']:.1%} (95% CI {low:.1%}-{high:.1%}), errors {s['errors']}")
for run in sorted(pathlib.Path("results").glob("run-*")):
    print(run.name)
    for s in json.loads((run / "summary.json").read_text()):
        ttft, itl = s["ttft_p95"] or 0, s["itl_p95"] or 0
        print(
            f"  {s['level']:>6} req/s  ok {s['ok']:>4}/{s['sent']:<4}"
            f"  TTFT p95 {ttft:6.3f} s  ITL p95 {itl:6.3f} s  goodput {s['goodput_ratio']:.0%}"
        )

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    "/content/relay-results-colab-all", "zip", "/content/relay", "results"
)
print(archive)
files.download(archive)